# Auction-Based Road Allocation — Interactive Simulation

**Research context.**  
Urban roads are a scarce shared resource.  When too many vehicles compete for the same segment at the same time,
congestion arises.  This notebook implements and compares *online auction mechanisms* for allocating road capacity
in a **time-expanded road network**, where each node is a (physical intersection, time-slot) pair.

**How it works.**  
Vehicles arrive sequentially, each carrying:
- an origin and destination,
- a preferred departure (or arrival) time,
- a maximum willingness to pay (`reserve`), and
- an urgency weight `alpha` that blends price and travel-time in the routing cost.

The mechanism allocates **complete routes** (bundles of segment–time pairs) via
shortest-path search on the time-expanded graph.  If a vehicle's shortest-path
cost exceeds its reserve, the vehicle is rejected.  Accepted vehicles are charged
the path cost and edge prices are updated for subsequent arrivals.

---

## Compared Strategies

| Strategy | Description |
|---|---|
| **Transport-Adapted Pricing** | Exponential price update with per-edge `vmax` scaled by demand/capacity and a travel-time component.  Designed to respect capacity constraints. |
| **Online Competitive** | Exponential update with a global parameter `r` and a global capacity scalar `s_max`.  Follows the BG-style online competitive framework (Buchbinder & Naor, 2009). |
| **Zero Pricing / Free Entry** | Prices are never updated.  Serves as a baseline: all feasible vehicles are accepted at zero toll. |
| **Static Median-Occupancy Pricing** | Runs the dynamic strategy internally for `capacity/2` allocations per edge, then freezes the price.  A static approximation to the dynamic rule. |
| **Smooth Tail** | Exponential on `[0, u0]`, cubic Hermite on `(u0, 1]`.  Reaches `vmax+1` at saturation for dual feasibility, while keeping price growth smooth near the capacity limit. |

## Output Metrics

- **Social welfare** — sum of (reserve − cost) for all served vehicles.
- **Service rate** — fraction of vehicles successfully allocated.
- **Avg travel time** — mean path length (slots) for served vehicles.
- **Avg delay** — mean entry and arrival delay relative to desired time.
- **Revenue** — total tolls collected.
- **Capacity utilisation** — fraction of segment capacity used per time slot.

---
## B · Setup

Run all cells in this section **before** anything else.  
If you are on a fresh Colab runtime, start with **B1** (clone the repo), then run **B2–B4** in order.

In [ ]:
# B1 — Clone or update the repository
import os

REPO_ROOT = "/content/transportation-auction-colab"

if not os.path.isdir(REPO_ROOT):
  from google.colab import userdata

  token = userdata.get("cloneCode")
  repo = "transportation-auction-colab"
  user = "CheckIT-App"

  !git clone https://{token}@github.com/{user}/{repo}.git
  #!git clone https://github.com/CheckIT-App/transportation-auction-colab.git {REPO_ROOT}
else:
    print("Repo already present — pulling latest updates ...")
    !git -C {REPO_ROOT} pull

In [ ]:
# B2 — Install required packages
!pip install -q -r {REPO_ROOT}/requirements.txt


In [ ]:
# B3 — Imports and path setup
import sys, os

REPO_ROOT = "/content/transportation-auction-colab"  # same as B1
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# All scripts must run from the repo root so relative paths (graph files, cache/) resolve correctly.
os.chdir(REPO_ROOT)

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 100

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

from ExpandedTimeSimulation.simulation_zefat.constants import ALL_STRATEGIES
from ExpandedTimeSimulation.simulation_zefat.colab_utils import (
    load_config_from_widgets,
    preview_network,
    generate_demand_preview,
    plot_demand_distribution,
    run_experiment,
    summarize_results,
    build_summary_tables,
    plot_results,
    export_results,
    decode_reject_reason,
    preview_network_map,
    preview_network_map_nolabels,
)

print("Setup complete.")


In [ ]:
# B4 — Mount Google Drive (optional but recommended for persistent cache and results)
# When mounted, graph files (.gpickle) and results (.csv/.xlsx) are stored in a
# dedicated Drive folder and survive Colab runtime resets and git pulls.
# If running locally, skip this cell.

import os

DRIVE_DATA_DIR = ""  # set below if Drive is available

try:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_DATA_DIR = "/content/drive/MyDrive/transportation_auction_data"
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

    print(f"Drive mounted. Persistent data folder:\n  {DRIVE_DATA_DIR}\n")
    print("Graph files, results, and FFLB caches saved here survive runtime resets and git pulls.")

    # List existing data files
    tracked_exts = (".gpickle", ".csv", ".xlsx", ".pkl")
    existing = sorted(
        f for f in os.listdir(DRIVE_DATA_DIR)
        if f.endswith(tracked_exts)
    )
    if existing:
        print(f"\nFound {len(existing)} existing file(s) in data folder:")
        for fname in existing:
            size_mb = os.path.getsize(os.path.join(DRIVE_DATA_DIR, fname)) / 1e6
            print(f"  {fname:<45s}  {size_mb:6.1f} MB")
    else:
        print("\nNo data files yet — the OSM graph will be downloaded on first run and saved here.")

    print("\nRun C1 now — widget defaults will point to your Drive folder automatically.")

except ImportError:
    print("Not running in Colab — Drive mount skipped. Using local paths.")


---
## C · Configuration

Adjust the sliders and fields below, then **run cell D1** (not this cell) to extract your settings.

> **Important:** Do not re-run *this* cell (C1) after changing sliders — re-running resets all widgets to their default values. Just move on to D1.

> **Tip — minimal first run:** keep `od_count ≤ 200`, `T ≤ 20`, `runs = 1`,
> and select only 2 strategies. Expected runtime on Colab free tier: ~60–90 s.

In [ ]:
# C1 — Configuration UI

import os

style = {"description_width": "200px"}
layout = widgets.Layout(width="480px")

# Resolve Drive-aware default paths (set by B4 if Drive is mounted)
_drive_dir = globals().get("DRIVE_DATA_DIR", "")
_graph_default   = os.path.join(_drive_dir, "har_nof.gpickle") if _drive_dir else "har_nof.gpickle"
_results_default = os.path.join(_drive_dir, "results.csv")     if _drive_dir else "results.csv"

if _drive_dir:
    print(f"Using Drive folder: {_drive_dir}")
    print(f"  graph_file  → {_graph_default}")
    print(f"  output dir  → {_drive_dir}  (filename auto-generated)\n")

# ── Basic parameters ─────────────────────────────────────────────────────
w = {
    # Runs / seed / output
    "num_runs":       widgets.IntSlider(value=1, min=1, max=10, step=1,
                          description="Number of runs", style=style, layout=layout),
    "base_seed":      widgets.IntText(value=2025,
                          description="Random seed", style=style, layout=layout),
    "excel_file":     widgets.Text(value=_results_default,
                          description="Output dir / file (.csv/.xlsx)", style=style, layout=layout),
    # Network
    "graph_file":     widgets.Text(value=_graph_default,
                          description="Graph file (.gpickle)", style=style, layout=layout),
    "place_name":     widgets.Text(value="Har Nof, Jerusalem, Israel",
                          description="OSM place name", style=style, layout=layout),
    # Demand file — populated by C2 after scanning Drive/local folder
    "demand_file":    widgets.Dropdown(
                          options=["(auto)"],
                          value="(auto)",
                          description="Demand file", style=style, layout=layout),
    # Time horizon
    "max_time_slots": widgets.IntSlider(value=20, min=10, max=200, step=5,
                          description="Network horizon T (slots)", style=style, layout=layout),
    "peak_slot":      widgets.IntSlider(value=10, min=1, max=100, step=1,
                          description="Peak demand slot", style=style, layout=layout),
    # Pricing
    "vmax":           widgets.FloatSlider(value=100.0, min=10.0, max=500.0, step=10.0,
                          description="vmax (price ceiling)", style=style, layout=layout),
    # Strategies
    "strategy_keys":  widgets.SelectMultiple(
                          options=ALL_STRATEGIES,
                          value=["Zero", "Transport-Adapted Pricing"],
                          description="Strategies", style=style,
                          layout=widgets.Layout(width="480px", height="120px")),
    # Vehicle mode
    "time_mode":      widgets.Dropdown(
                          options=["Both", "Only Entry", "Only Arrival"],
                          value="Both",
                          description="Vehicle mode", style=style, layout=layout),
    "arrival_percentage": widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                          description="Arrival fraction (Both)", style=style, layout=layout),
}

# ── Sweep parameters ─────────────────────────────────────────────────────
w_sweep = {
    "od_count_start": widgets.IntSlider(value=200, min=100, max=20000, step=100,
                          description="Vehicles: start", style=style, layout=layout),
    "od_count_end":   widgets.IntSlider(value=200, min=100, max=20000, step=100,
                          description="Vehicles: end", style=style, layout=layout),
    "od_count_step":  widgets.IntSlider(value=50, min=50, max=5000, step=50,
                          description="Vehicles: step", style=style, layout=layout),
    "peak_sigma_start": widgets.FloatSlider(value=5.0, min=0.5, max=20.0, step=0.5,
                          description="Sigma: start", style=style, layout=layout),
    "peak_sigma_end":   widgets.FloatSlider(value=5.0, min=0.5, max=20.0, step=0.5,
                          description="Sigma: end", style=style, layout=layout),
    "peak_sigma_step":  widgets.FloatSlider(value=1.0, min=0.5, max=10.0, step=0.5,
                          description="Sigma: step", style=style, layout=layout),
    "cap_factor_start": widgets.FloatSlider(value=100.0, min=10.0, max=300.0, step=10.0,
                          description="Capacity %: start", style=style, layout=layout),
    "cap_factor_end":   widgets.FloatSlider(value=100.0, min=10.0, max=300.0, step=10.0,
                          description="Capacity %: end", style=style, layout=layout),
    "cap_factor_step":  widgets.FloatSlider(value=10.0, min=5.0, max=100.0, step=5.0,
                          description="Capacity %: step", style=style, layout=layout),
}
w.update(w_sweep)

# ── Advanced parameters (hidden in accordion) ─────────────────────────────
w_adv = {
    "r":                 widgets.IntSlider(value=30, min=5, max=100, step=1,
                             description="r  (Online Competitive base)", style=style, layout=layout),
    "vehicle_T":         widgets.IntSlider(value=100, min=20, max=300, step=10,
                             description="vehicle_T  (max vehicle horizon)", style=style, layout=layout),
    "slot_seconds":      widgets.IntSlider(value=60, min=10, max=300, step=10,
                             description="Seconds per slot", style=style, layout=layout),
    "capacity_is_hourly":widgets.Checkbox(value=True,
                             description="Capacity is hourly", style=style),
    "smooth_tail_u0":    widgets.FloatSlider(value=0.95, min=0.5, max=1.0, step=0.01,
                             description="Smooth Tail u0", style=style, layout=layout),
    "path_solver":       widgets.Dropdown(
                             options=["astar_reverse_arrival", "dijkstra",
                                      "astar_fflb", "astar_euclidean",
                                      "bidirectional_dijkstra"],
                             value="astar_reverse_arrival",
                             description="Path solver", style=style, layout=layout),
    "alpha_lo":          widgets.FloatSlider(value=0.0, min=0.0, max=2.0, step=0.05,
                             description="Alpha: min", style=style, layout=layout),
    "alpha_hi":          widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.05,
                             description="Alpha: max", style=style, layout=layout),
    "entry_fee_lo":      widgets.FloatSlider(value=1.0, min=0.0, max=20.0, step=0.5,
                             description="Entry fee: min", style=style, layout=layout),
    "entry_fee_hi":      widgets.FloatSlider(value=5.0, min=0.0, max=20.0, step=0.5,
                             description="Entry fee: max", style=style, layout=layout),
    "lateness_fee_lo":   widgets.FloatSlider(value=1.0, min=0.0, max=20.0, step=0.5,
                             description="Lateness fee: min", style=style, layout=layout),
    "lateness_fee_hi":   widgets.FloatSlider(value=5.0, min=0.0, max=20.0, step=0.5,
                             description="Lateness fee: max", style=style, layout=layout),
    "reserve_lo":        widgets.FloatSlider(value=1.0, min=0.0, max=100.0, step=1.0,
                             description="Reserve price: min", style=style, layout=layout),
    "reserve_hi":        widgets.FloatSlider(value=100.0, min=1.0, max=500.0, step=5.0,
                             description="Reserve price: max", style=style, layout=layout),
    # ── Expected-demand baseline (ADVANCED) ───────────────────────────────
    # ⚠️ Changing these triggers a long re-run (5–30 min). See cell C2.
    "demand_runs":         widgets.IntSlider(value=3, min=1, max=20, step=1,
                               description="Demand baseline runs", style=style, layout=layout),
    "demand_num_vehicles": widgets.IntSlider(value=500, min=100, max=20000, step=100,
                               description="Vehicles per demand run", style=style, layout=layout),
    "demand_fraction":     widgets.FloatSlider(value=0.7, min=0.1, max=1.0, step=0.05,
                               description="Demand fraction (new area fallback)",
                               style=style, layout=layout),
    # ── Large-run controls ────────────────────────────────────────────────
    "verbose":             widgets.Checkbox(value=False,
                               description="Verbose output (per-run prints)", style=style),
    "resume":              widgets.Checkbox(value=False,
                               description="Resume (skip completed runs, CSV only)", style=style),
    "big_roads_only":      widgets.Checkbox(value=False,
                               description="Big roads only (motorway/primary/secondary)", style=style),
}
w.update(w_adv)

# ── Sweep validation label ────────────────────────────────────────────────
sweep_warning = widgets.HTML("")

def _check_sweep(*_):
    active = []
    if w["od_count_start"].value != w["od_count_end"].value:
        active.append("Vehicles")
    if w["peak_sigma_start"].value != w["peak_sigma_end"].value:
        active.append("Sigma")
    if w["cap_factor_start"].value != w["cap_factor_end"].value:
        active.append("Capacity %")
    if len(active) > 1:
        sweep_warning.value = (
            f"<span style='color:red'>&#9888; Only one sweep at a time "
            f"&#8212; active: {', '.join(active)}</span>"
        )
    elif len(active) == 1:
        sweep_warning.value = f"<span style='color:green'>&#10003; Sweep: {active[0]}</span>"
    else:
        sweep_warning.value = "<span style='color:grey'>Single run (no sweep)</span>"

_check_sweep()
for _k in ["od_count_start", "od_count_end",
           "peak_sigma_start", "peak_sigma_end",
           "cap_factor_start", "cap_factor_end"]:
    w[_k].observe(_check_sweep, names="value")

# ── Build UI ──────────────────────────────────────────────────────────────
advanced_box = widgets.Accordion(
    children=[widgets.VBox(list(w_adv.values()))],
    selected_index=None,
)
advanced_box.set_title(0, "Advanced settings")

basic_keys = [k for k in w if k not in w_sweep and k not in w_adv]
sweep_box = widgets.VBox(
    [widgets.HTML("<b>── Vehicles sweep</b>")]
    + [w["od_count_start"], w["od_count_end"], w["od_count_step"]]
    + [widgets.HTML("<b>── Peak sigma sweep</b>")]
    + [w["peak_sigma_start"], w["peak_sigma_end"], w["peak_sigma_step"]]
    + [widgets.HTML("<b>── Capacity factor sweep (% of network capacity)</b>")]
    + [w["cap_factor_start"], w["cap_factor_end"], w["cap_factor_step"]]
    + [widgets.HTML("<br>"), sweep_warning]
)
sweep_acc = widgets.Accordion(children=[sweep_box], selected_index=None)
sweep_acc.set_title(0, "Sweep / range settings")

ui = widgets.VBox(
    [widgets.HTML("<h4>Basic parameters</h4>")]
    + [w[k] for k in basic_keys]
    + [widgets.HTML("<br>"), sweep_acc]
    + [widgets.HTML("<br>"), advanced_box]
)
display(ui)
print("\nNote: output filename is auto-generated as {graph}_{N}veh_results.csv in the configured directory.")

### C2 · Expected-Demand Baseline

> ⚠️ **ADVANCED — long runtime (5–30 min).**  
> Generates the expected-demand file used by the simulation as a pricing reference.  
> **Runs only when `place_name` or `demand_num_vehicles` has changed** since the last run — otherwise skips automatically.  
> For a **new area** without a demand file, the simulation falls back to `demand_fraction × capacity` (set in Advanced settings above).

In [ ]:
# C2 — Expected-demand baseline
# Parquet files are ~10x smaller than Excel and load faster — preferred for upload/Colab.
# Run this cell to scan for / generate demand files and set config["expected_demand_file"].

import glob, os, re
from ExpandedTimeSimulation.simulation_zefat.experiments.batch_run import run_batch_for_demand

config = load_config_from_widgets(w)

_graph_stem  = os.path.splitext(config["graph_file"])[0]
_graph_name  = os.path.basename(_graph_stem)
_search_dir  = os.path.dirname(_graph_stem) or "."
_n_veh       = config["demand_num_vehicles"]

# Priority: parquet > csv > xlsx
def _resolve_demand_path(base_name):
    for ext in ["_edge_metrics.parquet", "_edge_metrics.csv", ".xlsx"]:
        p = os.path.join(_search_dir, base_name + ext)
        if os.path.exists(p):
            return p
    return None

# Scan existing demand files
_found = {}
for _f in glob.glob(os.path.join(_search_dir, f"{_graph_name}_demand_*veh_edge_metrics.parquet")):
    _base = re.sub(r"_edge_metrics\.parquet$", "", os.path.basename(_f))
    _found[_base] = _f
for _f in glob.glob(os.path.join(_search_dir, f"{_graph_name}_demand_*veh_edge_metrics.csv")):
    _base = re.sub(r"_edge_metrics\.csv$", "", os.path.basename(_f))
    if _base not in _found:
        _found[_base] = _f
for _f in glob.glob(os.path.join(_search_dir, f"{_graph_name}_demand_*veh.xlsx")):
    _base = re.sub(r"\.xlsx$", "", os.path.basename(_f))
    if _base not in _found:
        _found[_base] = _f

# Update dropdown
_OPT_AUTO = "(auto)"
_OPT_NONE = "(none — use demand_fraction fallback)"
_sorted_bases = sorted(_found.keys())
_new_options  = [_OPT_AUTO, _OPT_NONE] + _sorted_bases
_prev_sel = w["demand_file"].value
w["demand_file"].options = _new_options
w["demand_file"].value   = _prev_sel if _prev_sel in _new_options else _OPT_AUTO

if _found:
    print(f"Demand files found for '{_graph_name}':")
    for _base, _path in sorted(_found.items()):
        _mb  = os.path.getsize(_path) / 1e6 if os.path.exists(_path) else 0
        _fmt = os.path.splitext(_path)[1].upper().lstrip(".")
        _marker = " ◄ selected" if _base == w["demand_file"].value else ""
        print(f"  {_base}  ({_mb:.1f} MB, {_fmt}){_marker}")
else:
    print(f"No existing demand files for '{_graph_name}'.")

# Resolve / generate
_selected  = w["demand_file"].value
_auto_base = f"{_graph_name}_demand_{_n_veh}veh"
_auto_csv  = os.path.join(_search_dir, f"{_auto_base}.csv")
_auto_pq   = os.path.join(_search_dir, _auto_base)  # stem; parquet writer adds suffixes

if _selected == _OPT_NONE:
    config["expected_demand_file"] = None
    print("No demand file — simulation will use demand_fraction fallback.")

elif _selected == _OPT_AUTO:
    _resolved = _resolve_demand_path(_auto_base)
    if _resolved:
        _mb  = os.path.getsize(_resolved) / 1e6
        _fmt = os.path.splitext(_resolved)[1].upper().lstrip(".")
        config["expected_demand_file"] = _resolved
        print(f"✓ (auto) using existing: {os.path.basename(_resolved)}  ({_mb:.1f} MB, {_fmt})")
    else:
        print(f"Generating demand file → {_auto_csv}  ({_n_veh} vehicles, {config['demand_runs']} runs)")
        print("⚠️  This may take several minutes…")
        run_batch_for_demand(
            config["demand_runs"],
            place_name=config["place_name"],
            graph_file=config["graph_file"],
            od_count=_n_veh,
            excel_file=_auto_csv,
            parquet_file=_auto_pq,
            base_seed=config["base_seed"],
            slot_seconds=config["slot_seconds"],
            max_time_slots=config["max_time_slots"],
            vmax=config["vmax"],
            r=config["r"],
        )
        _pq_em  = _auto_pq + "_edge_metrics.parquet"
        _csv_em = os.path.join(_search_dir, f"{_auto_base}_edge_metrics.csv")
        if os.path.exists(_pq_em):
            config["expected_demand_file"] = _pq_em
        elif os.path.exists(_csv_em):
            config["expected_demand_file"] = _csv_em
        else:
            config["expected_demand_file"] = None
        print("✓ Done.")
        if _auto_base not in w["demand_file"].options:
            w["demand_file"].options = [_OPT_AUTO, _OPT_NONE, _auto_base] + _sorted_bases
            w["demand_file"].value   = _auto_base

else:
    config["expected_demand_file"] = _found.get(_selected)
    if config["expected_demand_file"]:
        print(f"Using selected: {os.path.basename(config['expected_demand_file'])}")
    else:
        print(f"WARNING: could not resolve '{_selected}' — falling back to None.")
        config["expected_demand_file"] = None

print(f"expected_demand_file = {config['expected_demand_file']}")


In [ ]:
# C3 — Load demand results for inspection (no re-run needed)
# Run this cell after C2 has generated the demand file, or any time later
# to reload it from disk.
import os, pandas as pd
from ExpandedTimeSimulation.simulation_zefat.constants import (
    SHEET_VEHICLES_TABLE, SHEET_EDGE_TIMESLICES,
)

_demand_path = config.get("expected_demand_file")
print(_demand_path)
if not _demand_path or not os.path.exists(_demand_path):
    print("No demand file loaded — run C2 first (or set config['expected_demand_file'] manually).")
    demand_veh_df = pd.DataFrame()
    demand_ts_df  = pd.DataFrame()
else:
    _stem = re.sub(r"_edge_metrics\.(csv|parquet)$", "", _demand_path)
    _ts_path   = _stem + "_edge_timeslices.parquet"
    _veh_path  = _stem + "_vehicles_table.parquet"
    _ts_csv    = _stem + "_edge_timeslices.csv"
    _veh_csv   = _stem + "_vehicles_table.csv"

    if os.path.exists(_ts_path):
        demand_ts_df = pd.read_parquet(_ts_path)
        print(f"edge_timeslices: {len(demand_ts_df):,} rows from {os.path.basename(_ts_path)}")
    elif os.path.exists(_ts_csv):
        demand_ts_df = pd.read_csv(_ts_csv)
        print(f"edge_timeslices: {len(demand_ts_df):,} rows from {os.path.basename(_ts_csv)}")
    elif _demand_path.endswith(".xlsx"):
        demand_ts_df = pd.read_excel(_demand_path, sheet_name=SHEET_EDGE_TIMESLICES)
        print(f"edge_timeslices: {len(demand_ts_df):,} rows (from Excel)")
    else:
        demand_ts_df = pd.DataFrame()
        print("edge_timeslices not found — re-run C2 to regenerate.")

    if os.path.exists(_veh_path):
        demand_veh_df = pd.read_csv(_veh_path) if _veh_path.endswith(".csv") else pd.read_parquet(_veh_path)
        print(f"vehicles_table:  {len(demand_veh_df):,} rows from {os.path.basename(_veh_path)}")
    elif os.path.exists(_veh_csv):
        demand_veh_df = pd.read_csv(_veh_csv)
        print(f"vehicles_table:  {len(demand_veh_df):,} rows from {os.path.basename(_veh_csv)}")
    elif _demand_path.endswith(".xlsx"):
        demand_veh_df = pd.read_excel(_demand_path, sheet_name=SHEET_VEHICLES_TABLE)
        print(f"vehicles_table:  {len(demand_veh_df):,} rows (from Excel)")
    else:
        demand_veh_df = pd.DataFrame()
        print("vehicles_table not found — re-run C2 to regenerate.")

In [ ]:
# C4 — Demand profile plots (temporal + spatial peak)
from ExpandedTimeSimulation.simulation_zefat.plots.plots import (
    plot_demand_entry_histogram,
    plot_demand_over_time,
)

_peak = config.get("peak_slot", 30)

# Temporal: when do vehicles enter? Shows peak shape.
if not demand_veh_df.empty:
    plot_demand_entry_histogram(demand_veh_df, peak_slot=_peak)
else:
    print("No vehicles_table — skipping entry histogram.")

if not demand_ts_df.empty:
    # Raw allocation counts per slot + top-10 busiest edges.
    plot_demand_over_time(demand_ts_df, metric="alloc_count", peak_slot=_peak, top_n_edges=10)
    # Utilisation (alloc_count / capacity) — shows which roads are crowded (>1 = over capacity).
    if "util" in demand_ts_df.columns:
        plot_demand_over_time(demand_ts_df, metric="util", peak_slot=_peak, top_n_edges=10)
    else:
        print("No 'util' column in edge_timeslices — skipping utilisation plot.")
else:
    print("No edge_timeslices — skipping demand-over-time plots.")

---
## D · Network Preparation

Loads the OSM road network from the `.gpickle` file specified above (or downloads it from OpenStreetMap on first run).
Displays a summary and map of the physical road network.

> **Caching.**  The time-expanded graph is cached automatically to `cache/expanded_net_<hash>.pkl`.
> On Colab, point `graph_file` to a path inside your mounted Drive so the cache persists between sessions.

In [ ]:
# D1 — Extract configuration from widgets
config = load_config_from_widgets(w)
print("Configuration loaded:")
for k, v in config.items():
    print(f"  {k:25s}: {v}")

In [ ]:
# D2 — Preview the road network (no time-expansion yet)
preview_network(config)

In [ ]:
# D3 — Interactive map (folium)
# folium is already installed via requirements.txt (B2), but re-installing is safe
road_map = preview_network_map(config)
display(road_map)


In [ ]:
# D4 — Interactive map without labels (for publication)
display(preview_network_map_nolabels(config))


---
## E · Demand Generation

Generates the synthetic vehicle fleet.  Each vehicle gets:
- a random origin–destination pair from the road network,
- a desired entry time drawn from a Gaussian centred on `peak_slot`,
- an `alpha` value (price–time urgency weight) drawn from a three-band mixture,
- a `reserve` price (willingness to pay) computed from alpha and path length.

The preview table shows the first 10 vehicles; the histograms show the full fleet distribution.

In [ ]:
# E1 — Generate vehicle fleet
vehicles, preview_df = generate_demand_preview(config)
print(f"Generated {len(vehicles):,} vehicles.\n")
display(preview_df)

In [ ]:
# E2 — Distribution plots
plot_demand_distribution(vehicles)

### E3 · Strategy Design — Theoretical

*How does each pricing rule behave mathematically?*

These plots are derived from the pricing formulae directly — no simulation data is needed. They give intuition for why strategies differ in their empirical acceptance rates and revenues below.

- **Pricing function shapes** — exponential, smooth-tail, and zero-pricing curves as a function of edge utilisation.
- **Transport-Adapted price growth** — how the recurrence $p_{k+1} = p_k \cdot e^{c/b} + \frac{bt}{T}(e^{c/b}-1)$ grows under different demand and urgency scenarios.

In [ ]:
# E3 — Pricing function shapes and Transport-Adapted price growth (no simulation data needed)
from ExpandedTimeSimulation.simulation_zefat.plots.plots import (
    plot_pricing_function_shapes,
    plot_transport_adapted_update_growth,
)

selected_strategies = list(config.get("strategy_keys", []))
plot_pricing_function_shapes(strategies=selected_strategies if selected_strategies else None)
plot_transport_adapted_update_growth()

---
## F · Simulation

Runs each selected strategy for the configured number of repetitions.
Progress is printed run-by-run.  Results are written to the Excel file
specified in the configuration.

> This cell may take **1–5 minutes** depending on `od_count`, `T`, and the number of strategies.
> Use the minimal configuration (≤ 200 vehicles, T ≤ 20, 2 strategies, 1 run) for a quick test.

In [ ]:
# F1 — Run experiment
excel_path = run_experiment(config)
print(f"\nResults saved to: {excel_path}")

---
## G · Results Summary

Loads all output sheets from the Excel file and builds per-metric comparison tables.
Green highlighting marks the best value per column.

In [ ]:
# G1 — Load result sheets
sheets = summarize_results(excel_path)

# Show a summary table of what was loaded
sheet_info = pd.DataFrame(
    [(name, len(df), len(df.columns)) for name, df in sheets.items()],
    columns=["Sheet", "Rows", "Columns"],
).set_index("Sheet")
display(sheet_info)

In [ ]:
# G2 — Per-metric comparison tables
tables = build_summary_tables(sheets)

for metric, df in tables.items():
    display(
        df.style
          .set_caption(metric)
          .highlight_max(subset=["Mean"], color="#c6efce")
          .format("{:.3f}")
    )


In [ ]:
# G3 — Rejection reason breakdown
from ExpandedTimeSimulation.simulation_zefat.constants import (
    SHEET_VEHICLES_TABLE, COL_STRATEGY, COL_REJECT_REASON
)

veh_df = sheets.get(SHEET_VEHICLES_TABLE, pd.DataFrame())

if not veh_df.empty and COL_REJECT_REASON in veh_df.columns:
    veh_df = veh_df.copy()
    veh_df["reject_label"] = veh_df[COL_REJECT_REASON].map(decode_reject_reason)
    breakdown = (
        veh_df.groupby([COL_STRATEGY, "reject_label"])
              .size()
              .unstack(fill_value=0)
    )
    breakdown.index.name = "Strategy"
    print("\nVehicle Outcome Breakdown")
    display(breakdown)
else:
    print("vehicles_table sheet is empty or missing reject_reason column.")

---
## H · Visualisations

Plots generated from the saved Excel file — re-run any group independently without re-running the simulation.

Results are organised into six groups, each preceded by a short explanation:

| Group | Contents |
|-------|----------|
| **H1 — Strategy Design** | Theoretical pricing function shapes and price growth |
| **H2 — Social Welfare Scaling** | Social welfare vs fleet size N |
| **H3 — Demand & Network Load** | Requests over time, edge hotspots, vehicles on road, utilisation |
| **H4 — Acceptance Dynamics** | Accepted/rejected per slot, acceptance rate, fee-band breakdown |
| **H5 — Price Dynamics** | Mean & peak edge prices over time per strategy |
| **H6 — Vehicle Economics** | Travel time vs α, delay vs fees, toll vs α, revenue |

> **Note:** Fleet composition histograms → **E2**.  Per-strategy summary tables and rejection counts → **G2–G3**.

### H3 · Social Welfare Scaling

*How efficiently does each strategy use road capacity as fleet size grows?*

G2 shows aggregate statistics at the simulated N. This chart sweeps across all N values in the results, revealing which mechanisms maintain high social welfare under increasing demand — and how much better they perform compared to the free-entry (Zero) baseline.

In [ ]:
# H4 — Social welfare vs N sweep
from ExpandedTimeSimulation.simulation_zefat.plots.plots import (
    plot_sw_sweep,
    compute_sw_time_only_from_df,
)

veh_df = sheets.get("vehicles_table", pd.DataFrame())
if veh_df.empty:
    print("vehicles_table is empty — run F1 first, then G1 to reload sheets.")
else:
    df_sw = compute_sw_time_only_from_df(
        veh_df,
        include_arrival_delay=True,
        include_entry_delay=False,
        served_only=True,
    )
    plot_sw_sweep(df_sw, baseline="Zero", compare="Transport-Adapted Pricing")

### H5 · Demand & Network Load

*When and where do vehicles compete for road capacity?*

- **Request timeline** — how many vehicles request each time slot under each strategy.
- **Edge heatmap** — which segments are persistent hotspots (top 20 busiest edges × time slots).
- **Vehicles on road** — concurrent active vehicles over the horizon, one line per strategy.
- **Utilisation distribution** — boxplot of edge load; shows whether pricing spreads or concentrates congestion.

In [ ]:
# H6 — Demand and network load
from ExpandedTimeSimulation.simulation_zefat.plots.plots import (
    plot_requests_over_time,
    plot_requests_heatmap,
    plot_mean_vehicles_on_road,
    plot_utilization_distribution,
)

ts_df = sheets.get("edge_timeslices", pd.DataFrame())

plot_requests_over_time(ts_df)
plot_requests_heatmap(ts_df, top_n=20)
plot_mean_vehicles_on_road(sheets["vehicles_table"])   # all strategies, auto-infers N
plot_utilization_distribution(ts_df)

### H7 · Acceptance & Rejection Dynamics

*Who gets accepted and when?*

G3 shows total rejection counts per strategy. These charts add the **temporal dimension**:
- Stacked counts of accepted / rejected (capacity) / rejected (reserve exceeded) per time slot
- Rolling acceptance rate — does selectivity increase as congestion builds over the horizon?
- Cumulative accepted and rejected — how quickly does each strategy saturate available capacity?
- Acceptance rate by lateness-fee band — are vehicles with tight time constraints systematically crowded out?

In [ ]:
# H8 — Acceptance and rejection dynamics over time
from ExpandedTimeSimulation.simulation_zefat.plots.plots import (
    plot_accepts_rejects_over_time,
    plot_acceptance_rate_over_time,
    plot_cumulative_accepts_rejects,
    plot_percent_accepted_over_time_by_fee_bands_5000,
)
from ExpandedTimeSimulation.simulation_zefat.constants import COL_LATENESS_FEE

veh_df = sheets["vehicles_table"]

plot_accepts_rejects_over_time(veh_df, by="request")
plot_acceptance_rate_over_time(veh_df, by="request")
plot_cumulative_accepts_rejects(veh_df, by="request")
plot_percent_accepted_over_time_by_fee_bands_5000(
    veh_df,
    bands=((1, 2), (4, 5)),
    panels="bands",
    fee_col=COL_LATENESS_FEE,
    mode="arrival",
)

### H9 · Price Dynamics

*How do edge prices evolve over the simulation horizon?*

Mean and peak prices per time slot, averaged across runs, reveal whether Transport-Adapted pricing smoothly tracks congestion or whether any strategy lets prices spike at peak demand.

In [ ]:
# H10 — Price evolution per strategy
from ExpandedTimeSimulation.simulation_zefat.plots.plots import plot_price_evolution_per_strategy

ts_df = sheets.get("edge_timeslices", pd.DataFrame())

if not ts_df.empty and "N" in ts_df.columns:
    n_vals = pd.to_numeric(ts_df["N"], errors="coerce").dropna()
    common_N = int(n_vals.value_counts().idxmax()) if not n_vals.empty else None
    if common_N is not None:
        plot_price_evolution_per_strategy(ts_df, N_value=common_N, exclude_zero=True)
    else:
        print("Could not infer N — skipping price evolution plot.")
else:
    print("edge_timeslices unavailable or missing N column — skipping.")

### H11 · Vehicle Economics

*What do individual vehicles experience?*

- **Travel time vs α** — do high-urgency vehicles (large α, prefer time over price) end up with shorter routes?
- **Arrival delay vs lateness fee** — does paying a higher penalty actually reduce arrival delay?
- **Toll vs travel time** — do vehicles that pay more in road tolls travel faster?
- **Revenue comparison** — total road toll collected per strategy across fleet sizes (visual companion to G2).
- **Toll vs α** — do urgent vehicles end up on more expensive edges? Consistent across strategies?

> `paid_fee` = **road toll only** (sum of edge prices along the path). Entry/lateness delay fees are routing weights, not monetary payments.

In [ ]:
# H12 — Vehicle economics
from ExpandedTimeSimulation.simulation_zefat.plots.plots import (
    plot_travel_time_vs_alpha_by_N,
    plot_delay_or_arrival_vs_fee_by_N,
    plot_price_vs_travel_time_by_N,
    plot_revenue_comparison,
    plot_toll_vs_alpha,
)
from ExpandedTimeSimulation.simulation_zefat.constants import COL_PAID_FEE, COL_TRAVEL_TIME

veh_df = sheets["vehicles_table"]
Ns = tuple(sorted(pd.to_numeric(veh_df["N"], errors="coerce").dropna().unique().astype(int)))

plot_travel_time_vs_alpha_by_N(veh_df, Ns=Ns, bins=12)
plot_delay_or_arrival_vs_fee_by_N(veh_df, pair="arrival", Ns=Ns)
plot_price_vs_travel_time_by_N(veh_df, Ns=Ns, price_col=COL_PAID_FEE, travel_time_col=COL_TRAVEL_TIME)
plot_revenue_comparison(veh_df)
plot_toll_vs_alpha(veh_df, Ns=Ns)

### H13 · Custom X/Y Plot

Choose any X and Y metric to compare strategies on a single chart.  
**Sweep-based X axes** (Vehicles, Sigma, Capacity) only work when the last simulation used a sweep — the merged Excel must contain `sweep_param` / `sweep_value` columns.  
**Binned X axes** (Alpha, Entry/Arrival delay cost) are automatically divided into `n_bins` equal-width bins; the line is plotted at each bin midpoint.  
When `num_runs > 1` a ±1 std shaded band is shown.

In [ ]:
# H14 — Custom plot config

X_OPTIONS = {
    "Time slot":           "time_slot",
    "Vehicles (sweep)":    "vehicles_sweep",
    "Sigma (sweep)":       "sigma_sweep",
    "Capacity % (sweep)":  "capacity_sweep",
    "Alpha":               "alpha",
    "Entry delay cost":    "entry_delay_cost",
    "Arrival delay cost":  "arrival_delay_cost",
}
Y_OPTIONS = {
    "Social welfare":      "social_welfare",
    "% Acceptance":        "acceptance",
    "Avg price per route": "avg_price",
    "Avg entry delay":     "avg_entry_delay",
    "Avg arrival delay":   "avg_arrival_delay",
    "Speed":               "speed",
    "Travel time":         "travel_time",
}

w_plot = {
    "x_metric": widgets.Dropdown(
                    options=list(X_OPTIONS),
                    value="Time slot",
                    description="X axis:", style=style, layout=layout),
    "y_metric": widgets.Dropdown(
                    options=list(Y_OPTIONS),
                    value="% Acceptance",
                    description="Y axis:", style=style, layout=layout),
    "n_bins":   widgets.IntSlider(value=10, min=3, max=30, step=1,
                    description="Bins (alpha/cost):", style=style, layout=layout),
    "save_dest": widgets.Dropdown(
                    options=["None", "Download", "Save to Drive", "Both"],
                    value="None",
                    description="Save CSV:", style=style, layout=layout),
    "drive_dir": widgets.Text(
                    value="/content/drive/MyDrive/",
                    description="Drive folder:", style=style, layout=layout),
}
display(widgets.VBox([widgets.HTML("<b>Custom plot settings</b>")] + list(w_plot.values())))


In [ ]:
# H15 — Run custom plot
from ExpandedTimeSimulation.simulation_zefat.colab_utils import plot_custom

_x_key = X_OPTIONS[w_plot["x_metric"].value]
_y_key = Y_OPTIONS[w_plot["y_metric"].value]

_summary_df = plot_custom(
    excel_file=config.get("excel_file", "results.xlsx"),
    x_metric=_x_key,
    y_metric=_y_key,
    strategies=list(config.get("strategy_keys", [])) or None,
    n_bins=w_plot["n_bins"].value,
)

if _summary_df is not None:
    from ExpandedTimeSimulation.simulation_zefat.colab_utils import _X_LABELS, _Y_LABELS
    _x_label = _X_LABELS.get(_x_key, _x_key).lower().replace(" ", "_")
    _y_label = _Y_LABELS.get(_y_key, _y_key).lower().replace(" ", "_")
    _csv_name = f"custom_plot_{_x_label}_vs_{_y_label}.csv"
    _dest = w_plot["save_dest"].value

    if _dest != "None":
        try:
            import tempfile, os
            from google.colab import files

            if _dest in ("Save to Drive", "Both"):
                _drive_path = os.path.join(w_plot["drive_dir"].value.rstrip("/"), _csv_name)
                _summary_df.to_csv(_drive_path, index=False)
                print(f"Saved to Drive: {_drive_path}")

            if _dest in ("Download", "Both"):
                _tmp = os.path.join(tempfile.gettempdir(), _csv_name)
                _summary_df.to_csv(_tmp, index=False)
                files.download(_tmp)

        except ImportError:
            print(f"Not running in Colab — displaying DataFrame (save manually as {_csv_name}):")
            display(_summary_df)


---
## I · Export

Saves each result sheet as a CSV file and writes the experiment configuration as JSON.
The Excel file is already written by the simulation step.

In [ ]:
# I1 — Export CSVs + config JSON
export_results(excel_path, config, output_dir="experiment_output")

---
## J · Developer / Advanced Tools

Low-level inspection cells.  Useful for debugging, understanding individual vehicle
allocations, tracing edge prices, or benchmarking path solvers.  These cells are
independent — run them in any order after the simulation has completed.

In [ ]:
# J1 — Inspect a single vehicle allocation
from ExpandedTimeSimulation.simulation_zefat.constants import (
    SHEET_VEHICLES_TABLE, COL_STRATEGY, COL_SERVED,
    COL_ALPHA, COL_RESERVE, COL_PAID_FEE,
    COL_TRAVEL_TIME, COL_ENTRY_DELAY, COL_ARRIVAL_DELAY,
    COL_REJECT_REASON,
)

VEHICLE_IDX = 0        # <-- change this to inspect a different vehicle
STRATEGY    = None     # <-- set to a strategy name string, or None for the first available

veh_df = sheets.get(SHEET_VEHICLES_TABLE, pd.DataFrame())
if veh_df.empty:
    print("vehicles_table is empty — run F1 first.")
else:
    strats = veh_df[COL_STRATEGY].unique()
    chosen_strat = STRATEGY if STRATEGY in strats else strats[0]
    subset = veh_df[veh_df[COL_STRATEGY] == chosen_strat]
    row = subset.iloc[VEHICLE_IDX]
    print(f"Vehicle #{VEHICLE_IDX}  —  strategy: {chosen_strat}")
    print(f"  served          : {row.get(COL_SERVED)}")
    print(f"  reject reason   : {decode_reject_reason(row.get(COL_REJECT_REASON, 0))}")
    print(f"  alpha           : {row.get(COL_ALPHA, '—'):.3f}")
    print(f"  reserve         : {row.get(COL_RESERVE, '—'):.3f}")
    print(f"  paid fee        : {row.get(COL_PAID_FEE, '—')}")
    print(f"  travel time     : {row.get(COL_TRAVEL_TIME, '—')} slots")
    print(f"  entry delay     : {row.get(COL_ENTRY_DELAY, '—')} slots")
    print(f"  arrival delay   : {row.get(COL_ARRIVAL_DELAY, '—')} slots")

In [ ]:
# J2 — Inspect price evolution for a single edge over time
from ExpandedTimeSimulation.simulation_zefat.constants import (
    SHEET_EDGE_TIMESLICES, COL_STRATEGY, COL_T, COL_PRICE, COL_UTIL, COL_EDGE
)

EDGE_IDX = 0    # <-- index into the list of unique edges

ts_df = sheets.get(SHEET_EDGE_TIMESLICES, pd.DataFrame())
if ts_df.empty:
    print("edge_timeslices is empty — run F1 first.")
else:
    unique_edges = ts_df[COL_EDGE].unique()
    if EDGE_IDX >= len(unique_edges):
        print(f"EDGE_IDX={EDGE_IDX} out of range. There are {len(unique_edges)} unique edges.")
    else:
        chosen_edge = unique_edges[EDGE_IDX]
        subset = ts_df[ts_df[COL_EDGE] == chosen_edge]

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        for strat, grp in subset.groupby(COL_STRATEGY):
            grp_sorted = grp.sort_values(COL_T)
            axes[0].plot(grp_sorted[COL_T], grp_sorted[COL_PRICE], label=strat, marker="o", ms=3)
            axes[1].plot(grp_sorted[COL_T], grp_sorted[COL_UTIL],  label=strat, marker="o", ms=3)

        axes[0].set_title(f"Price over time — edge {EDGE_IDX}", fontsize=12)
        axes[0].set_xlabel("Time slot"); axes[0].set_ylabel("Price")
        axes[0].legend(fontsize=9)

        axes[1].set_title(f"Utilisation over time — edge {EDGE_IDX}", fontsize=12)
        axes[1].set_xlabel("Time slot"); axes[1].set_ylabel("Utilisation")
        axes[1].axhline(1.0, color="red", linestyle="--", linewidth=0.8, label="capacity")
        axes[1].legend(fontsize=9)

        plt.tight_layout()
        plt.show()
        print(f"Edge: {chosen_edge}")

In [ ]:
# J3 — Profile runtime for a minimal run
import time

PROFILE_CONFIG = {
    "od_count": 100,
    "num_runs": 1,
    "max_time_slots": 15,
    "vmax": 100.0,
    "r": 30,
    "slot_seconds": 60,
    "vehicle_T": 50,
    "peak_slot": 8,
    "peak_sigma": 3.0,
    "strategy_keys": ["Zero", "Transport-Adapted Pricing"],
    "base_seed": 42,
    "excel_file": "_profile_run.xlsx",
    "graph_file": config.get("graph_file", "har_nof.gpickle"),
    "place_name": config.get("place_name", "Har Nof, Jerusalem, Israel"),
    "capacity_is_hourly": True,
    "smooth_tail_u0": 0.95,
}

t0 = time.perf_counter()
run_experiment(PROFILE_CONFIG)
elapsed = time.perf_counter() - t0
print(f"\nProfiling complete: {elapsed:.1f}s  ({PROFILE_CONFIG['od_count']} vehicles, "
      f"T={PROFILE_CONFIG['max_time_slots']}, {len(PROFILE_CONFIG['strategy_keys'])} strategies)")

In [ ]:
# J4 — Path-solver benchmark across graph sizes and fleet sizes
#
# Compares all solvers on multiple graphs (har_nof, yokneam, zefat, ashdod).
# FFLB cache is built once per graph and reused across fleet sizes and runs.
# Outputs: timing table + line chart (ms/veh vs N, one panel per graph).
#
# Edit GRAPHS / VEHICLE_COUNTS / T below to customise.

import copy, os, time
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from ExpandedTimeSimulation.simulation_zefat.auction_simulator import (
    AuctionSimulator, precompute_fflb, load_fflb,
)
from ExpandedTimeSimulation.simulation_zefat.network import TimeExpandedRoadNetwork
from ExpandedTimeSimulation.simulation_zefat.strategy_factory import make_strategy
from ExpandedTimeSimulation.simulation_zefat.experiments.batch_run import _load_or_build_graph_and_xy
from ExpandedTimeSimulation.simulation_zefat.experiments.vehicle_generation import (
    mixed_alpha_sampler, assign_peak_desired_entry, PeakSchedule,
)

# ── Widget controls ────────────────────────────────────────────────────────
_style  = {"description_width": "180px"}
_layout = widgets.Layout(width="460px")

_GRAPH_OPTIONS = {
    "Har Nof  (48 nodes)":   {"file": "har_nof.gpickle",  "place": "Har Nof, Jerusalem, Israel"},
    "Yokneam  (582 nodes)":  {"file": "yokneam.gpickle",  "place": "Yokneam, Israel"},
    "Zefat    (769 nodes)":  {"file": "zefat.gpickle",    "place": "Safed, Israel"},
    "Ashdod   (2706 nodes)": {"file": "ashdod.gpickle",   "place": "Ashdod, Israel"},
}

w_bench = {
    "graphs": widgets.SelectMultiple(
        options=list(_GRAPH_OPTIONS),
        value=["Har Nof  (48 nodes)", "Zefat    (769 nodes)"],
        description="Graphs to test", style=_style,
        layout=widgets.Layout(width="460px", height="100px"),
    ),
    "n_min":  widgets.IntSlider(value=50,  min=10,  max=500,  step=10,
                  description="Fleet size: min", style=_style, layout=_layout),
    "n_max":  widgets.IntSlider(value=300, min=50,  max=2000, step=50,
                  description="Fleet size: max", style=_style, layout=_layout),
    "n_step": widgets.IntSlider(value=125, min=10,  max=500,  step=10,
                  description="Fleet size: step", style=_style, layout=_layout),
    "T":      widgets.IntSlider(value=20,  min=10,  max=100,  step=5,
                  description="Time horizon T", style=_style, layout=_layout),
    "seed":   widgets.IntText(value=42, description="Random seed", style=_style, layout=_layout),
}
display(widgets.VBox(
    [widgets.HTML("<b>Benchmark settings</b> — adjust, then run the next cell")]
    + list(w_bench.values())
))


In [ ]:
# J4b — Run benchmark and plot results
# Run J4 first to set parameters, then run this cell.

SOLVERS = [
    "dijkstra",
    "bidirectional_dijkstra",
    "astar_euclidean",
    "astar_fflb",
    "astar_reverse_arrival",
]
SOLVER_COLORS = {
    "dijkstra":               "#2980b9",
    "bidirectional_dijkstra": "#e74c3c",
    "astar_euclidean":        "#8e44ad",
    "astar_fflb":             "#e67e22",
    "astar_reverse_arrival":  "#27ae60",
}
SLOT_SECONDS = 60

selected_graphs  = list(w_bench["graphs"].value)
vehicle_counts   = list(range(w_bench["n_min"].value,
                               w_bench["n_max"].value + 1,
                               w_bench["n_step"].value))
T_val   = w_bench["T"].value
SEED    = w_bench["seed"].value

print(f"Graphs : {selected_graphs}")
print(f"Fleets : {vehicle_counts}")
print(f"T={T_val}, seed={SEED}\n")

rows = []

for graph_label in selected_graphs:
    gcfg  = _GRAPH_OPTIONS[graph_label]
    gfile = gcfg["file"]
    gname = graph_label.split()[0].lower()

    if not os.path.exists(gfile):
        print(f"[SKIP] {gfile} not found — upload it to the Colab runtime first.")
        continue

    print(f"\n{'='*55}")
    print(f"Graph: {graph_label}  ({gfile})")
    print(f"{'='*55}")

    node_xy_file = os.path.splitext(gfile)[0] + "_node_xy_meters.pkl"
    loader, node_xy_file = _load_or_build_graph_and_xy(
        place_name=gcfg["place"],
        graph_file=gfile,
        node_xy_file=node_xy_file,
        time_slot_duration=SLOT_SECONDS,
        od_count=max(vehicle_counts),
    )
    loader.enrich_graph()
    base_edges = loader.convert_to_base_edges()
    n_nodes = loader.G.number_of_nodes()

    # Build / load FFLB cache once per graph
    fflb_file = os.path.splitext(gfile)[0] + "_fflb.pkl"
    fflb_cache = load_fflb(fflb_file)
    if fflb_cache is None:
        print(f"  Computing FFLB cache -> {fflb_file}")
        _net_tmp = TimeExpandedRoadNetwork(
            base_edges, max_time_slots=T_val, vmax=100.0, r=30,
            pricing_strategy=make_strategy("Zero"),
            capacity_is_hourly=True, slot_seconds=SLOT_SECONDS,
            node_xy_file=node_xy_file,
        )
        fflb_cache = precompute_fflb(_net_tmp, fflb_file)
        print(f"  FFLB ready ({len(fflb_cache['by_src'])} nodes)")
    else:
        print(f"  FFLB loaded from {fflb_file}")

    for n_veh in vehicle_counts:
        import random
        random.seed(SEED)
        loader.od_count = n_veh
        loader.generate_od_pairs()
        alpha_mix = mixed_alpha_sampler(
            [(0.4, (0.0, 0.3)), (0.4, (0.3, 0.7)), (0.2, (0.7, 1.0))]
        )
        vehicles = loader.generate_vehicles(alpha=alpha_mix)
        assign_peak_desired_entry(
            vehicles,
            schedule=PeakSchedule(peak_slot=T_val // 2, sigma=3.0, horizon_T=T_val * 3),
            write_arrival=True,
        )
        print(f"\n  Fleet size: {n_veh}")

        for solver in SOLVERS:
            cache_arg = fflb_cache if solver in (
                "astar_fflb", "astar_fflb_delay", "astar_reverse_arrival"
            ) else None
            try:
                net = TimeExpandedRoadNetwork(
                    base_edges, max_time_slots=T_val, vmax=100.0, r=30,
                    pricing_strategy=make_strategy("Zero"),
                    capacity_is_hourly=True, slot_seconds=SLOT_SECONDS,
                    node_xy_file=node_xy_file,
                )
                sim = AuctionSimulator(net, copy.deepcopy(vehicles),
                                       path_solver=solver, fflb_cache=cache_arg)
                t0 = time.perf_counter()
                sim.run()
                elapsed = time.perf_counter() - t0
                ms_pv = elapsed / n_veh * 1000
                label = solver + (" [cached]" if cache_arg else "")
                print(f"    {label:42s}  {elapsed:6.2f}s  ({ms_pv:6.1f} ms/veh)")
                rows.append({
                    "graph": graph_label, "n_nodes": n_nodes,
                    "n_vehicles": n_veh, "solver": solver,
                    "cached": cache_arg is not None,
                    "time_s": round(elapsed, 3),
                    "ms_per_veh": round(ms_pv, 2),
                })
            except Exception as e:
                print(f"    {solver:42s}  ERROR: {e}")

# ── Results table ──────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
if df.empty:
    print("No results — check that graph files are uploaded.")
else:
    df["solver_label"] = df.apply(
        lambda r: r["solver"] + (" [cached]" if r.get("cached") else ""), axis=1
    )
    pivot = df.pivot_table(
        index="solver_label",
        columns=["graph", "n_vehicles"],
        values="ms_per_veh",
        aggfunc="mean",
    ).round(1)
    print("\n\n=== ms / vehicle ===")
    display(pivot.style
            .highlight_min(axis=0, color="#c6efce")
            .format("{:.1f}"))

    # ── Bar chart: first chosen graph, ms/veh per solver grouped by fleet size ──
    first_graph = selected_graphs[0]
    df_bar = df[df["graph"] == first_graph].copy()

    if not df_bar.empty:
        fleet_sizes = sorted(df_bar["n_vehicles"].unique())
        x = np.arange(len(fleet_sizes))
        bar_width = 0.8 / len(SOLVERS)

        fig_bar, ax_bar = plt.subplots(figsize=(max(8, len(fleet_sizes) * 2), 5))

        for i, solver in enumerate(SOLVERS):
            sub = df_bar[df_bar["solver"] == solver]
            vals = [
                sub[sub["n_vehicles"] == n]["ms_per_veh"].mean() if not sub[sub["n_vehicles"] == n].empty else 0
                for n in fleet_sizes
            ]
            cached = not sub.empty and sub["cached"].iloc[0]
            lbl = solver + (" [cached]" if cached else "")
            bars = ax_bar.bar(
                x + i * bar_width, vals,
                width=bar_width,
                color=SOLVER_COLORS.get(solver, "#888"),
                label=lbl,
                edgecolor="white", linewidth=0.5,
                alpha=0.9 if solver == "astar_reverse_arrival" else 0.75,
            )
            # Highlight astar_reverse_arrival bars with a bold edge
            if solver == "astar_reverse_arrival":
                for bar in bars:
                    bar.set_edgecolor("#1a5c35")
                    bar.set_linewidth(1.5)

        ax_bar.set_xticks(x + bar_width * (len(SOLVERS) - 1) / 2)
        ax_bar.set_xticklabels([str(n) for n in fleet_sizes], fontsize=10)
        ax_bar.set_xlabel("Fleet size (vehicles)", fontsize=11)
        ax_bar.set_ylabel("ms / vehicle", fontsize=11)
        n_nodes_bar = df_bar["n_nodes"].iloc[0]
        ax_bar.set_title(
            f"Solver comparison — {first_graph}  ({n_nodes_bar} nodes, T={T_val})\n"
            "Lower is faster",
            fontsize=12,
        )
        ax_bar.legend(fontsize=9, loc="upper right")
        ax_bar.grid(axis="y", alpha=0.3)
        ax_bar.set_axisbelow(True)
        plt.tight_layout()
        plt.savefig("solver_benchmark_bar.png", dpi=120, bbox_inches="tight")
        print(f"\nBar chart saved to solver_benchmark_bar.png  ({first_graph})")
        plt.show()

    # ── Line chart: ms/veh vs fleet size, one panel per graph ─────────────
    graphs_in_data = df["graph"].unique()
    ncols = min(2, len(graphs_in_data))
    nrows = int(np.ceil(len(graphs_in_data) / ncols))

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(7 * ncols, 4.5 * nrows), squeeze=False)

    for idx, gname in enumerate(graphs_in_data):
        ax = axes[idx // ncols][idx % ncols]
        sub = df[df["graph"] == gname].dropna(subset=["ms_per_veh"])
        n_nodes = sub["n_nodes"].iloc[0] if not sub.empty else "?"

        for solver in SOLVERS:
            s = sub[sub["solver"] == solver].sort_values("n_vehicles")
            if s.empty:
                continue
            cached = s["cached"].iloc[0]
            lbl = solver + (" [cached]" if cached else "")
            lw  = 2.5 if solver == "astar_reverse_arrival" else 1.5
            ls  = "-"  if solver == "astar_reverse_arrival" else "--"
            ax.plot(s["n_vehicles"], s["ms_per_veh"],
                    marker="o", label=lbl,
                    color=SOLVER_COLORS.get(solver, "#888"),
                    linewidth=lw, linestyle=ls, markersize=5)

        ax.set_title(f"{gname}\n({n_nodes} nodes, T={T_val})", fontsize=11)
        ax.set_xlabel("Fleet size (vehicles)", fontsize=10)
        ax.set_ylabel("ms / vehicle", fontsize=10)
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(True, alpha=0.3)

    for idx in range(len(graphs_in_data), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle("Path-solver runtime comparison  (lower = faster)", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig("solver_benchmark.png", dpi=120, bbox_inches="tight")
    print("\nLine chart saved to solver_benchmark.png")
    plt.show()


---
## K · Route Visualisation

Runs a small simulation (Zero pricing, 30 vehicles) and plots the **first allocated route** on the physical road map.  
Route arcs are drawn in **red**; the rest of the network is grey.  
The **green star** marks the origin; the **blue square** marks the destination.

In [ ]:
# K1 — Visualise a random served route on the Har Nof (or configured place) map
#
# Uses sheets["vehicles_table"] from G1 — no new simulation needed.
# Each run picks a different randomly served vehicle.
# Shows two maps:
#   • Static matplotlib — abstract road network, route highlighted in red
#   • Interactive Folium — OpenStreetMap base with full road network (styled as D3),
#                          route follows actual road geometry in red on top
import pickle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import folium
from IPython.display import display as ipy_display

from ExpandedTimeSimulation.simulation_zefat.constants import (
    SHEET_VEHICLES_TABLE, COL_STRATEGY, COL_SERVED,
    COL_ALPHA, COL_TRAVEL_TIME,
)

# ── 1. Pull a random served vehicle ──────────────────────────────────────
veh_df = sheets.get(SHEET_VEHICLES_TABLE, None)
if veh_df is None or veh_df.empty:
    print("ERROR: 'sheets' is empty — run cells F1 and G1 first.")
else:
    first_strat = veh_df[COL_STRATEGY].iloc[0]
    served = veh_df[
        (veh_df[COL_STRATEGY] == first_strat) &
        (veh_df[COL_SERVED] == True)
    ]

    if served.empty:
        print(f"No served vehicles for strategy '{first_strat}'.")
    else:
        row       = served.sample(1).iloc[0]          # ← random pick each run
        src_node  = int(row["source"])
        dst_node  = int(row["destination"])
        travel_tt = row.get(COL_TRAVEL_TIME)
        alpha_v   = float(row.get(COL_ALPHA, float("nan")))
        veh_idx   = row.name

        print(f"Strategy      : {first_strat}")
        print(f"Vehicle index : {veh_idx}  (random from {len(served)} served vehicles)")
        print(f"Route         : {src_node}  →  {dst_node}")
        print(f"  travel time : {travel_tt} slot(s)")
        print(f"  alpha       : {alpha_v:.3f}")

        # ── 2. Load the base OSM graph ────────────────────────────────────
        GRAPH_FILE = config.get("graph_file", "har_nof.gpickle")
        with open(GRAPH_FILE, "rb") as fh:
            G_base = pickle.load(fh)

        # ── 3. Compute shortest path ──────────────────────────────────────
        weight = "travel_time" if nx.get_edge_attributes(G_base, "travel_time") else None
        route_nodes = []
        try:
            route_nodes = nx.shortest_path(G_base, src_node, dst_node, weight=weight)
            print(f"Physical path : {len(route_nodes)} nodes, {len(route_nodes)-1} arc(s)")
        except (nx.NetworkXNoPath, nx.NodeNotFound) as err:
            print(f"Cannot compute path: {err}")

        if route_nodes:
            route_edges = {
                (route_nodes[i], route_nodes[i+1]) for i in range(len(route_nodes)-1)
            }

            # ── 4. Coordinate-system-aware helpers ────────────────────────
            _sample = next(iter(G_base.nodes(data=True)))[1]
            _sx, _sy = float(_sample.get("x", 0)), float(_sample.get("y", 0))
            _in_deg  = abs(_sx) <= 180 and abs(_sy) <= 90
            print(f"Graph coords  : sample x={_sx:.4f}, y={_sy:.4f}  "
                  f"({'degrees ✓' if _in_deg else 'projected metres — converting'})")

            def _latlon(node_id):
                """(lat, lon) from a node, handling degrees or projected metres."""
                d = G_base.nodes.get(node_id, {})
                x, y = float(d.get("x", 0)), float(d.get("y", 0))
                if abs(x) <= 180 and abs(y) <= 90:
                    return y, x
                try:
                    import pyproj
                    crs = G_base.graph.get("crs") or "EPSG:32636"
                    lon, lat = pyproj.Transformer.from_crs(
                        crs, "EPSG:4326", always_xy=True
                    ).transform(x, y)
                    return float(lat), float(lon)
                except Exception:
                    return None, None

            def _edge_latlon_coords(u, v):
                """[[lat,lon], ...] for an edge using road geometry, or straight-line fallback."""
                edge_data = G_base.get_edge_data(u, v)
                if edge_data:
                    ed = next(iter(edge_data.values()))
                    geom = ed.get("geometry")
                    if geom is not None:
                        raw = list(geom.coords)
                        if _in_deg:
                            return [[y, x] for x, y in raw]
                        try:
                            import pyproj
                            crs = G_base.graph.get("crs") or "EPSG:32636"
                            tf = pyproj.Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
                            return [[lat, lon] for lon, lat in (tf.transform(x, y) for x, y in raw)]
                        except Exception:
                            pass
                ll_u, ll_v = _latlon(u), _latlon(v)
                if ll_u[0] is not None and ll_v[0] is not None:
                    return [list(ll_u), list(ll_v)]
                return []

            # ── 5. Static matplotlib plot (abstract road network) ─────────
            pos = {
                n: (d["x"], d["y"])
                for n, d in G_base.nodes(data=True)
                if "x" in d and "y" in d
            }

            fig, ax = plt.subplots(figsize=(13, 11))
            ax.set_facecolor("#f0f3f4")
            fig.patch.set_facecolor("#f0f3f4")

            for u, v in G_base.edges():
                if u in pos and v in pos:
                    ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]],
                            color="#aab7b8", linewidth=0.6, alpha=0.65, zorder=1)
            for u, v in route_edges:
                pu, pv = pos.get(u), pos.get(v)
                if pu and pv:
                    ax.plot([pu[0], pv[0]], [pu[1], pv[1]],
                            color="#e74c3c", linewidth=4.0, alpha=0.95, zorder=3,
                            solid_capstyle="round", solid_joinstyle="round")
            mid = [n for n in route_nodes if n not in (src_node, dst_node) and n in pos]
            if mid:
                ax.scatter([pos[n][0] for n in mid], [pos[n][1] for n in mid],
                           c="#e74c3c", s=14, zorder=4, alpha=0.85)
            if src_node in pos:
                ax.scatter(*pos[src_node], c="#27ae60", s=300, marker="*", zorder=6)
            if dst_node in pos:
                ax.scatter(*pos[dst_node], c="#2980b9", s=200, marker="s", zorder=6)

            ax.legend(handles=[
                mpatches.Patch(color="#e74c3c", label=f"Route arcs ({len(route_edges)})"),
                mpatches.Patch(color="#aab7b8", label="Road network"),
                plt.Line2D([0],[0], marker="*", color="w", markerfacecolor="#27ae60",
                           markersize=14, label=f"Origin ({src_node})"),
                plt.Line2D([0],[0], marker="s", color="w", markerfacecolor="#2980b9",
                           markersize=10, label=f"Destination ({dst_node})"),
            ], loc="upper right", fontsize=11, framealpha=0.95, edgecolor="#bdc3c7")
            ax.set_title(
                f"Random served route  [{first_strat}]  —  vehicle #{veh_idx}\n"
                f"α = {alpha_v:.3f}  |  travel time = {travel_tt} slot(s)  |"
                f"  {len(route_edges)} arc(s) highlighted",
                fontsize=13, pad=12)
            ax.axis("off")
            plt.tight_layout()
            plt.show()

            # ── 6. Interactive Folium map — same style as D3 labeled map ──
            # Start from preview_network_map so the basemap and blue road network match D3.
            fmap = preview_network_map(config)

            route_layer = folium.FeatureGroup(name="Route", overlay=True, show=True)
            for u, v in zip(route_nodes[:-1], route_nodes[1:]):
                coords = _edge_latlon_coords(u, v)
                if coords:
                    folium.PolyLine(
                        coords,
                        color="#e74c3c", weight=7, opacity=0.95,
                        tooltip=f"{u} → {v}",
                    ).add_to(route_layer)
            route_layer.add_to(fmap)

            src_ll = _latlon(src_node)
            if src_ll[0] is not None:
                folium.Marker(list(src_ll), popup=f"Origin — node {src_node}",
                              icon=folium.Icon(color="green", icon="play")).add_to(fmap)

            dst_ll = _latlon(dst_node)
            if dst_ll[0] is not None:
                folium.Marker(list(dst_ll), popup=f"Destination — node {dst_node}",
                              icon=folium.Icon(color="blue", icon="stop")).add_to(fmap)

            folium.LayerControl().add_to(fmap)
            print(f"\nDone — {len(route_edges)} arc(s) highlighted in red.")
            ipy_display(fmap)